# Orquestador final de evaluación — Colab, Kaggle y local

Ejecuta el Bloque C de la issue #9: instala el paquete, corre pruebas y benchmark, valida filtro → métrica con datos sintéticos y permite evaluar las seis condiciones reales. Importa módulos de producción; no duplica algoritmos.

## 1. Plataforma y configuración

El modo synthetic funciona sin GPU ni credenciales. El modo real requiere EVALUATION_MODEL_PATHS_JSON, EVALUATION_IMAGES_DIR y EVALUATION_GROUND_TRUTH_CSV. Kaggle puede usar /kaggle/input; Colab copia datos de Drive al SSD de /content.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path


def detect_platform():
    forced = os.getenv("EVALUATION_PLATFORM", "").strip().lower()
    if forced in {"colab", "kaggle", "local"}:
        return forced
    if os.getenv("COLAB_RELEASE_TAG") or "google.colab" in sys.modules:
        return "colab"
    if os.getenv("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/working").exists():
        return "kaggle"
    return "local"


PLATFORM = detect_platform()
WORK_ROOT = Path("/content") if PLATFORM == "colab" else (
    Path("/kaggle/working") if PLATFORM == "kaggle" else Path.cwd()
)
OUTPUT_DIR = Path(os.getenv(
    "EVALUATION_OUTPUT_DIR", str(WORK_ROOT / "evaluation_output")
))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if PLATFORM == "colab" and os.getenv("EVALUATION_MOUNT_DRIVE", "0") == "1":
    from google.colab import drive

    drive.mount("/content/drive")

print(f"Plataforma: {PLATFORM} | salida local: {OUTPUT_DIR}")


## 2. Sparse clone e instalación editable

Si estamos dentro del repositorio se reutiliza el checkout. En la nube se clona únicamente experiments con historial superficial. EVALUATION_REPO_REF selecciona una rama publicada.

In [ ]:
REPO_URL = os.getenv(
    "EVALUATION_REPO_URL",
    "https://github.com/unsa-semester-2026-A/ia_article.git",
)
REPO_REF = os.getenv("EVALUATION_REPO_REF", "main")
current = Path.cwd().resolve()

if current.name == "experiments" and (current / "pyproject.toml").exists():
    EXPERIMENTS_DIR = current
elif (current / "experiments" / "pyproject.toml").exists():
    EXPERIMENTS_DIR = current / "experiments"
else:
    repository_dir = WORK_ROOT / "ia_article"
    if not repository_dir.exists():
        subprocess.run(
            [
                "git", "clone", "--depth", "1", "--filter=blob:none",
                "--sparse", "--branch", REPO_REF, REPO_URL, str(repository_dir),
            ],
            check=True,
        )
        subprocess.run(
            ["git", "sparse-checkout", "set", "experiments"],
            cwd=repository_dir,
            check=True,
        )
    EXPERIMENTS_DIR = repository_dir / "experiments"

os.chdir(EXPERIMENTS_DIR)
if os.getenv("EVALUATION_SKIP_INSTALL", "0") != "1":
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".[cloud]"],
        check=True,
    )
print("Paquete editable listo en", EXPERIMENTS_DIR)


## 3. Pruebas, benchmark y smoke test

El pipeline se detiene si una prueba falla. Los reportes se guardan primero en el disco local de la VM.

In [ ]:
def run_and_save(command, report_name):
    completed = subprocess.run(
        command, cwd=EXPERIMENTS_DIR, text=True, capture_output=True
    )
    report_path = OUTPUT_DIR / report_name
    report_path.write_text(
        completed.stdout + completed.stderr, encoding="utf-8"
    )
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f"Falló el comando; revisar {report_path}")
    return report_path


TEST_REPORT = run_and_save(
    [sys.executable, "-m", "pytest", "src/evaluation/", "-q"],
    "evaluation_tests.txt",
)
BENCHMARK_REPORT = run_and_save(
    [
        sys.executable, "-m", "pytest",
        "src/evaluation/test_metric.py::test_metric_benchmark_50k_predictions_10k_gt",
        "-q", "-s",
    ],
    "metric_benchmark.txt",
)

from src.evaluation.pipeline import run_synthetic_pipeline, write_evaluation_report

synthetic_result = run_synthetic_pipeline()
assert abs(synthetic_result.macro_score - 1.0 / 9.0) < 1e-12
assert synthetic_result.motion_by_clip["clip_demo"].removed_predictions == 12
SYNTHETIC_REPORT = write_evaluation_report(
    OUTPUT_DIR / "synthetic_evaluation.json",
    {"synthetic": synthetic_result},
)
print("Pipeline sintético correcto | Macro AP =", synthetic_result.macro_score)


## 4. Evaluación real opcional

Solo corre con EVALUATION_RUN_MODE=real. Deben configurarse exactamente Base 0, Base 1, Base 2, Mejora A, Mejora B y Mejora C. Usa conf=0.001, convierte clases 0..8 a 1..9 y procesa cada clip secuencialmente.

In [ ]:
RUN_MODE = os.getenv("EVALUATION_RUN_MODE", "synthetic").strip().lower()
EXPECTED_CONDITIONS = {
    "Base 0", "Base 1", "Base 2",
    "Mejora A", "Mejora B", "Mejora C",
}
MODEL_PATHS = json.loads(os.getenv("EVALUATION_MODEL_PATHS_JSON", "{}"))
IMAGES_SOURCE = Path(os.getenv("EVALUATION_IMAGES_DIR", ""))
default_gt = (
    "/content/drive/MyDrive/ia_article/00_raw/train.csv"
    if PLATFORM == "colab"
    else ""
)
GROUND_TRUTH_CSV = Path(
    os.getenv("EVALUATION_GROUND_TRUTH_CSV", default_gt)
)


def stage_images_on_fast_disk(source):
    if PLATFORM != "colab" or not str(source).startswith("/content/drive/"):
        return source
    destination = OUTPUT_DIR / "validation_images"
    if not destination.exists():
        shutil.copytree(source, destination)
    return destination


REAL_REPORT = None
if RUN_MODE == "real":
    if set(MODEL_PATHS) != EXPECTED_CONDITIONS:
        raise ValueError("MODEL_PATHS debe contener exactamente seis condiciones")
    if not IMAGES_SOURCE.is_dir():
        raise FileNotFoundError(f"Directorio inexistente: {IMAGES_SOURCE}")
    if not GROUND_TRUTH_CSV.is_file():
        raise FileNotFoundError(f"CSV GT inexistente: {GROUND_TRUTH_CSV}")
    for model_path in MODEL_PATHS.values():
        if not Path(model_path).is_file():
            raise FileNotFoundError(f"Pesos inexistentes: {model_path}")

    import cv2
    from ultralytics import YOLO
    from src.evaluation.pipeline import (
        evaluate_dataset, infer_clip, load_ground_truth_csv, split_frame_id,
    )

    images_dir = stage_images_on_fast_disk(IMAGES_SOURCE)
    image_paths_by_clip = {}
    for image_path in sorted(images_dir.rglob("*")):
        if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        frame_id = image_path.stem
        clip_id, _ = split_frame_id(frame_id)
        image_paths_by_clip.setdefault(clip_id, {})[frame_id] = image_path
    if not image_paths_by_clip:
        raise RuntimeError("No se encontraron imágenes de validación")

    selected_frame_ids = {
        frame_id
        for clip_paths in image_paths_by_clip.values()
        for frame_id in clip_paths
    }
    ground_truths = load_ground_truth_csv(
        GROUND_TRUTH_CSV, selected_frame_ids
    )
    condition_results = {}
    for condition_name in sorted(EXPECTED_CONDITIONS):
        print("Evaluando", condition_name)
        model = YOLO(MODEL_PATHS[condition_name])
        predictions_by_clip = {}
        homographies_by_clip = {}
        for clip_id, frame_paths in sorted(image_paths_by_clip.items()):
            frames = {}
            for frame_id, image_path in sorted(frame_paths.items()):
                image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
                if image is None:
                    raise ValueError(f"No se pudo leer {image_path}")
                frames[frame_id] = image
            predictions, homographies = infer_clip(model, frames)
            predictions_by_clip[clip_id] = predictions
            homographies_by_clip[clip_id] = homographies
        condition_results[condition_name] = evaluate_dataset(
            predictions_by_clip, homographies_by_clip, ground_truths
        )

    REAL_REPORT = write_evaluation_report(
        OUTPUT_DIR / "six_conditions_evaluation.json",
        condition_results,
    )
    for name, result in condition_results.items():
        print(f"{name}: Macro AP-rIoU={result.macro_score:.6f}")
else:
    print("Modo synthetic: se omite inferencia real.")


## 5. Drive API opcional y finalización

Solo sube reportes pequeños con EVALUATION_DRIVE_TOKEN_PATH y EVALUATION_DRIVE_FOLDER_ID. Sin ambas variables no intenta autenticarse. La desconexión de Colab es opcional.

In [ ]:
def upload_report_if_configured(report_path):
    token_path = Path(os.getenv("EVALUATION_DRIVE_TOKEN_PATH", ""))
    folder_id = os.getenv("EVALUATION_DRIVE_FOLDER_ID", "").strip()
    if not folder_id or not token_path.is_file():
        print("Drive API no configurada; reporte local:", report_path)
        return None

    from google.oauth2.credentials import Credentials
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload

    credentials = Credentials.from_authorized_user_file(
        str(token_path),
        ["https://www.googleapis.com/auth/drive.file"],
    )
    service = build("drive", "v3", credentials=credentials)
    mimetype = (
        "application/json" if report_path.suffix == ".json" else "text/plain"
    )
    uploaded = service.files().create(
        body={"name": report_path.name, "parents": [folder_id]},
        media_body=MediaFileUpload(str(report_path), mimetype=mimetype),
        fields="id",
        supportsAllDrives=True,
    ).execute()
    print("Reporte subido con ID", uploaded["id"])
    return uploaded["id"]


reports_to_sync = [TEST_REPORT, BENCHMARK_REPORT, SYNTHETIC_REPORT]
if REAL_REPORT is not None:
    reports_to_sync.append(REAL_REPORT)
for report in reports_to_sync:
    upload_report_if_configured(report)

print("Evaluación terminada. Reportes en", OUTPUT_DIR)
if (
    PLATFORM == "colab"
    and os.getenv("EVALUATION_DISCONNECT_COLAB", "0") == "1"
):
    from google.colab import runtime

    runtime.unassign()
